# NB-09: VAE Encoder -- 4-Level Spatial Downsampling

## Learning Objectives
- Trace Encoder3d's 4-level spatial downsampling path from 64x64 to 8x8
- Understand that temporal dimension is preserved without feat_cache (Pitfall 2)
- Perform the reparameterization trick (mu/log_var split + sampling)
- Know the production temporal compression formula and why it requires chunked inference

## Prerequisites
- **Prior notebooks:** NB-08 (CausalConv3d, ResidualBlock, AttentionBlock)
- **Assumed concepts:** Spatial downsampling, VAE reparameterization trick

## Concept Map
- **Encoder3d** -> produces latents consumed by DiT (Phase 2) and decoded by Decoder3d (NB-10)
- **Reparameterization** -> connects encoder to the probabilistic latent space
- **Latent normalization** -> final step before DiT consumption (Phase 6 Integration)

In [ ]:
import sys
import types
import importlib.util
import pathlib

_here = pathlib.Path().resolve()
PROJECT_ROOT = None
for _candidate in [_here] + list(_here.parents)[:6]:
    if (_candidate / "diffsynth" / "models" / "wan_video_dit.py").exists():
        PROJECT_ROOT = _candidate
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find diffsynth/models/wan_video_dit.py. "
        "Run this notebook from Course/ inside the project checkout."
    )
print(f"Project root: {PROJECT_ROOT}")

_cam_stub = types.ModuleType("diffsynth.models.wan_video_camera_controller")
_cam_stub.SimpleAdapter = type("SimpleAdapter", (), {"__init__": lambda s, *a, **kw: None})
sys.modules["diffsynth"] = types.ModuleType("diffsynth")
sys.modules["diffsynth.models"] = types.ModuleType("diffsynth.models")
sys.modules["diffsynth.models.wan_video_camera_controller"] = _cam_stub

_vae_path = PROJECT_ROOT / "diffsynth" / "models" / "wan_video_vae.py"
_spec = importlib.util.spec_from_file_location("diffsynth.models.wan_video_vae", _vae_path)
vae_mod = importlib.util.module_from_spec(_spec)
sys.modules["diffsynth.models.wan_video_vae"] = vae_mod
_spec.loader.exec_module(vae_mod)

from diffsynth.models.wan_video_vae import (
    Encoder3d, Decoder3d, CausalConv3d, ResidualBlock, AttentionBlock, Resample, RMS_norm
)
import torch
import torch.nn as nn
import torch.nn.functional as F

print("Setup complete.")

## Data Flow Diagram

```
Encoder3d Forward -- 4-Level Spatial Downsample
================================================

VIDEO [B, 3, T, H, W]  (input: 1, 3, 5, 64, 64)
    |
    v
conv1: CausalConv3d(3, dim=32, k=3, p=1)
    |                                        [1, 32, 5, 64, 64]
    v
+-- Level 0: ResBlock(32->32) x2 + Resample(downsample2d)
|                                            [1, 32, 5, 32, 32]  (spatial /2)
+-- Level 1: ResBlock(32->64) x2 + Resample(downsample3d*)
|                                            [1, 64, 5, 16, 16]  (spatial /2)
+-- Level 2: ResBlock(64->128) x2 + Resample(downsample3d*)
|                                            [1, 128, 5, 8, 8]   (spatial /2)
+-- Level 3: ResBlock(128->128) x2 [NO downsample]
    |                                        [1, 128, 5, 8, 8]
    v
MIDDLE: ResBlock(128) -> AttentionBlock(128) -> ResBlock(128)
    |                                        [1, 128, 5, 8, 8]
    v
HEAD: RMS_norm -> SiLU -> CausalConv3d(128, z_dim*2=8, k=3, p=1)
    |                                        [1, 8, 5, 8, 8]
    v
ENCODER OUTPUT [B, z_dim*2, T, H/8, W/8]

* downsample3d in non-cached mode: spatial-only (temporal preserved)
  See "Production Temporal Compression" section below.
```

## 1. Encoder3d Constructor -- Sub-Module Inventory

The Encoder3d constructor builds a flat `ModuleList` of layers organized into 4 levels, a middle block, and a head projection. Each level contributes `num_res_blocks` ResidualBlocks + 1 Resample (except the last level).

| Sub-module | Type | Config | Role |
|------------|------|--------|------|
| `conv1` | CausalConv3d | 3->32, k=3, p=1 | RGB input projection |
| `downsamples[0:3]` | ResBlock x2 + Resample | Level 0 (downsample2d) | Spatial /2 |
| `downsamples[3:6]` | ResBlock x2 + Resample | Level 1 (downsample3d) | Spatial /2 |
| `downsamples[6:9]` | ResBlock x2 + Resample | Level 2 (downsample3d) | Spatial /2 |
| `downsamples[9:11]` | ResBlock x2 (no Resample) | Level 3 | Final processing |
| `middle` | [ResBlock, AttentionBlock, ResBlock] | dim=128 | Global spatial context |
| `head` | [RMS_norm, SiLU, CausalConv3d] | 128 -> z_dim*2=8 | Output projection |

Note: `downsamples` is a flat `nn.Sequential`: each level contributes `num_res_blocks` ResidualBlocks + 1 Resample (except last level). Total layers in downsamples: 2+1 + 2+1 + 2+1 + 2 = 11.

Source: `wan_video_vae.py`, lines 517-567

In [ ]:
# Source: wan_video_vae.py, lines 517-567
# VAE reduced config
dim = 32
z_dim = 4
dim_mult = [1, 2, 4, 4]
num_res_blocks = 2

# Note: In production, VideoVAE_ passes z_dim*2 to Encoder3d so the encoder
# outputs 2*z_dim channels for the mu/log_var split (see wan_video_vae.py, line 971).
# We follow the same pattern here.
enc = Encoder3d(dim=dim, z_dim=z_dim * 2, dim_mult=dim_mult,
                num_res_blocks=num_res_blocks,
                temperal_downsample=[False, True, True])

total_params = sum(p.numel() for p in enc.parameters())
print(f"Encoder3d parameters: {total_params:,}")
print(f"\nConfig: dim={dim}, z_dim={z_dim}, dim_mult={dim_mult}")
print(f"Channel progression: {[dim * m for m in dim_mult]}")
print(f"Encoder z_dim arg: z_dim*2 = {z_dim*2} (so head outputs {z_dim*2} channels for mu/log_var split)")
print(f"\nSub-module breakdown:")
print(f"  conv1: {sum(p.numel() for p in enc.conv1.parameters()):,} params")
print(f"  downsamples ({len(enc.downsamples)} layers): {sum(p.numel() for p in enc.downsamples.parameters()):,} params")
print(f"  middle ({len(enc.middle)} layers): {sum(p.numel() for p in enc.middle.parameters()):,} params")
print(f"  head: {sum(p.numel() for p in enc.head.parameters()):,} params")

## 2. Step-by-Step Forward Pass Trace

We now trace a single forward pass through the encoder, printing shapes at each stage. Watch how spatial dimensions halve at each Resample while temporal dimension stays constant.

In [ ]:
# Source: wan_video_vae.py, lines 569-617
# Step-by-step Encoder3d forward pass trace

enc.eval()
x = torch.randn(1, 3, 5, 64, 64)
print(f"Input: {x.shape}  [B, C_rgb, T, H, W]")
print(f"{'='*60}")

# Step 1: conv1 -- RGB to dim channels
with torch.no_grad():
    x = enc.conv1(x)
assert x.shape == torch.Size([1, 32, 5, 64, 64])
print(f"Step 1 -- conv1 (3->32):     {x.shape}")

# Step 2: Level 0 -- ResBlock x2 + downsample2d
for layer in list(enc.downsamples)[:3]:  # 2 ResBlocks + 1 Resample
    with torch.no_grad():
        x = layer(x)
assert x.shape == torch.Size([1, 32, 5, 32, 32])
print(f"Step 2 -- Level 0 + ds2d:    {x.shape}  (spatial /2: 64->32)")

# Step 3: Level 1 -- ResBlock x2 + downsample3d (spatial only, no cache)
for layer in list(enc.downsamples)[3:6]:
    with torch.no_grad():
        x = layer(x)
assert x.shape == torch.Size([1, 64, 5, 16, 16])
print(f"Step 3 -- Level 1 + ds3d:    {x.shape}  (spatial /2: 32->16, channels 32->64)")

# Step 4: Level 2 -- ResBlock x2 + downsample3d (spatial only, no cache)
for layer in list(enc.downsamples)[6:9]:
    with torch.no_grad():
        x = layer(x)
assert x.shape == torch.Size([1, 128, 5, 8, 8])
print(f"Step 4 -- Level 2 + ds3d:    {x.shape}  (spatial /2: 16->8, channels 64->128)")

# Step 5: Level 3 -- ResBlock x2 (no downsample)
for layer in list(enc.downsamples)[9:]:
    with torch.no_grad():
        x = layer(x)
assert x.shape == torch.Size([1, 128, 5, 8, 8])
print(f"Step 5 -- Level 3 (no ds):   {x.shape}  (shape preserved)")

# Step 6: Middle block
for layer in enc.middle:
    with torch.no_grad():
        x = layer(x)
assert x.shape == torch.Size([1, 128, 5, 8, 8])
print(f"Step 6 -- Middle block:      {x.shape}  (ResBlock + Attn + ResBlock)")

# Step 7: Head (norm + SiLU + conv to z_dim*2)
with torch.no_grad():
    for layer in enc.head:
        x = layer(x)
assert x.shape == torch.Size([1, 8, 5, 8, 8])
print(f"Step 7 -- Head (128->8):     {x.shape}  (z_dim*2 = 8 channels)")
print(f"{'='*60}")
print(f"\nTemporal dim: 5 throughout (PRESERVED without feat_cache)")
print(f"Spatial compression: 64x64 -> 8x8 (8x in each dimension)")
print(f"Channel expansion: 3 -> 8 (z_dim*2 for mu/log_var)")

### Shape Trace Summary

| Step | Operation | Output Shape | Spatial | Temporal | Channels |
|------|-----------|-------------|---------|----------|----------|
| Input | -- | [1, 3, 5, 64, 64] | 64x64 | 5 | 3 (RGB) |
| 1 | conv1 | [1, 32, 5, 64, 64] | 64x64 | 5 | 32 |
| 2 | Level 0 + ds2d | [1, 32, 5, 32, 32] | 32x32 | 5 | 32 |
| 3 | Level 1 + ds3d | [1, 64, 5, 16, 16] | 16x16 | 5 | 64 |
| 4 | Level 2 + ds3d | [1, 128, 5, 8, 8] | 8x8 | 5 | 128 |
| 5 | Level 3 | [1, 128, 5, 8, 8] | 8x8 | 5 | 128 |
| 6 | Middle | [1, 128, 5, 8, 8] | 8x8 | 5 | 128 |
| 7 | Head | [1, 8, 5, 8, 8] | 8x8 | 5 | 8 (z_dim*2) |

In [ ]:
# Verify: full forward pass matches step-by-step trace
x_full = torch.randn(1, 3, 5, 64, 64)
with torch.no_grad():
    out_full = enc(x_full)

assert out_full.shape == torch.Size([1, 8, 5, 8, 8]), f"Expected [1,8,5,8,8], got {out_full.shape}"
print(f"Full encoder forward: {x_full.shape} -> {out_full.shape}")
print(f"Compression ratio:")
print(f"  Spatial: 64x64 -> 8x8 = 64x compression per frame")
print(f"  Channels: 3 -> 8 (z_dim*2)")
print(f"  Total voxels: {3*5*64*64:,} -> {8*5*8*8:,} = {3*5*64*64 / (8*5*8*8):.1f}x compression")

## 3. Temporal Dimension: Why T Stays Constant

The `temperal_downsample=[False, True, True]` parameter *suggests* temporal downsampling at levels 1 and 2. But the Resample module's `downsample3d` mode only applies its temporal `time_conv` when `feat_cache is not None` (see `wan_video_vae.py`, lines 162-173).

Without `feat_cache` (our notebook mode), Resample `downsample3d` performs **spatial-only downsampling** -- it reshapes to `(B*T, C, H, W)`, applies a 2D stride-2 convolution, then reshapes back to 5D. The `time_conv` is never called.

This is by design: the network is architecturally spatial-only. Temporal compression is a property of how the chunked inference loop feeds frames to the encoder, not of the network itself. See the "Production Temporal Compression" section below for the formula.

In [ ]:
# Verify: temporal dimension is preserved at EVERY stage (no feat_cache)
print("Temporal dimension verification (no feat_cache):")
print(f"  Input T = 5")

x_t = torch.randn(1, 3, 5, 64, 64)
with torch.no_grad():
    # Check after each Resample (the layers that COULD do temporal downsample)
    x_t = enc.conv1(x_t)
    print(f"  After conv1:      T = {x_t.shape[2]}")

    for i, layer in enumerate(enc.downsamples):
        x_t = layer(x_t)
        if hasattr(layer, 'mode'):  # This is a Resample layer
            print(f"  After Resample {i}: T = {x_t.shape[2]}  (mode={layer.mode})")

    print(f"  Final T = {x_t.shape[2]}")

assert x_t.shape[2] == 5, f"Temporal dim should be 5, got {x_t.shape[2]}"
print(f"\nConfirmed: T=5 preserved throughout (downsample3d is spatial-only without feat_cache)")

## 4. Reparameterization: From Encoder Output to Latent z

The encoder outputs `z_dim*2` channels -- the first half is **mu** (mean) and the second half is **log_var** (log variance). The reparameterization trick samples a latent `z` that is differentiable with respect to both `mu` and `log_var`:

```
z = mu + std * eps
where: std = exp(0.5 * log_var)
       eps ~ N(0, 1)  (random noise)
```

The gradient flows through `mu` and `log_var` (not through the random sample `eps`), making the sampling step differentiable for training.

In the actual codebase (`VideoVAE_`, line 971), Encoder3d is created with `z_dim * 2` as its output channels. Then `conv1` (a learned 1x1 CausalConv3d) projects the encoder output, and `.chunk(2, dim=1)` splits into `mu` and `log_var`. In our reduced demo, we skip the intermediate `conv1` and split the encoder output directly.

Source: `wan_video_vae.py`, lines 978-1039 (VideoVAE_.forward, reparameterize)

In [ ]:
# Source: wan_video_vae.py, lines 978-1039 (VideoVAE_.forward, reparameterize)
# Demonstrate the full encode -> reparameterize pipeline

# Start from encoder output
x_input = torch.randn(1, 3, 5, 64, 64)
with torch.no_grad():
    encoder_out = enc(x_input)

print(f"Encoder output: {encoder_out.shape}  (z_dim*2 = {z_dim*2} channels)")

# Split into mu and log_var
mu, log_var = encoder_out.chunk(2, dim=1)
print(f"mu:      {mu.shape}  (first {z_dim} channels)")
print(f"log_var: {log_var.shape}  (last {z_dim} channels)")

# Reparameterization trick
# Source: wan_video_vae.py, lines 1036-1039
std = torch.exp(0.5 * log_var)
eps = torch.randn_like(std)
z = eps * std + mu

assert z.shape == torch.Size([1, z_dim, 5, 8, 8])
print(f"\nLatent z: {z.shape}  [B, z_dim={z_dim}, T=5, H/8=8, W/8=8]")
print(f"\nReparameterization trick:")
print(f"  std = exp(0.5 * log_var)  ->  shape {std.shape}")
print(f"  eps ~ N(0, 1)             ->  shape {eps.shape}")
print(f"  z = eps * std + mu        ->  shape {z.shape}")
print(f"\nDeterministic mode (for inference): z = mu (skip sampling)")

## 5. Latent Normalization

The `WanVideoVAE` wrapper (line 1058) applies per-channel normalization after encoding. It stores fixed `mean` and `std` tensors (shape `[z_dim]`) computed from training data statistics.

**Normalization (encode path, before DiT):**
```python
z_normalized = (z - mean) * (1.0 / std)
```

**Denormalization (decode path, before decoder):**
```python
z = z_normalized * std + mean
```

This ensures latents have approximately zero mean and unit variance -- important for the DiT which expects well-conditioned inputs. The normalization is perfectly invertible, so no information is lost.

Source: `wan_video_vae.py`, lines 1058-1073

In [ ]:
# Source: wan_video_vae.py, lines 1058-1073 (WanVideoVAE)
# Demonstrate latent normalization (simplified -- using random mean/std for demo)

# In production: mean and std are loaded from checkpoint (16 values each for z_dim=16)
# For our reduced config (z_dim=4), simulate with random values
mean = torch.randn(1, z_dim, 1, 1, 1)  # broadcastable over [B, z_dim, T, H, W]
std = torch.rand(1, z_dim, 1, 1, 1) + 0.5  # positive std

# Normalize (encode path -- before DiT)
z_normalized = (z - mean) * (1.0 / std)
print(f"Before normalization - z stats: mean={z.mean():.3f}, std={z.std():.3f}")
print(f"After normalization  - z stats: mean={z_normalized.mean():.3f}, std={z_normalized.std():.3f}")
print(f"Shape preserved: {z_normalized.shape}")

# Denormalize (decode path -- before decoder)
z_recovered = z_normalized * std + mean
diff = (z_recovered - z).abs().max().item()
assert diff < 1e-5, f"Normalization round-trip failed: max diff = {diff}"
print(f"\nRound-trip verification: max diff = {diff:.2e} (normalize -> denormalize)")
print(f"Confirmed: normalization is perfectly invertible")

## 6. Production Temporal Compression (Conceptual)

In production, the encoder processes video in chunks with `feat_cache` enabled. This is the only way temporal compression occurs -- the network architecture itself is purely spatial.

**Chunked encoding process:**
- First chunk: 1 frame -> produces 1 latent frame
- Subsequent chunks: 4 frames each -> produces 1 latent frame each
- Formula: `T_latent = 1 + (T_video - 1) // 4`

```
Production chunked encoding (T=21 video frames):

Video frames: [F0] [F1 F2 F3 F4] [F5 F6 F7 F8] [F9 F10 F11 F12] [F13 F14 F15 F16] [F17 F18 F19 F20]
              chunk0   chunk1        chunk2          chunk3            chunk4             chunk5
                |         |              |               |                 |                  |
                v         v              v               v                 v                  v
Latent frames: [L0]     [L1]           [L2]           [L3]             [L4]               [L5]

T_latent = 1 + (21-1)//4 = 6
Temporal compression: 21 -> 6 = 3.5x
```

In our notebooks, we call the encoder directly without `feat_cache`, so T is preserved. This conceptual section explains how the pipeline achieves temporal compression in production.

Source: `wan_video_vae.py`, lines 984-1000 (VideoVAE_.encode chunked loop)

## 7. Note: 3.8B VAE Variant

The WAN codebase also contains `Encoder3d_38` / `Decoder3d_38` (lines 620, 842), a larger variant with `dim=160`, `z_dim=48` that uses a patchify front-end and `AvgDown3D`/`DupUp3D` for downsampling instead of strided convolutions. This achieves higher compression (`upsampling_factor=16` vs 8) but the standard WAN 2.2 pipeline (`WanVideoVAE`) uses the original architecture we studied above.

---

## Summary

### Key Takeaways
- **Encoder3d** performs 3 levels of spatial 2x downsampling: 64x64 -> 8x8 (8x per axis)
- **Temporal dimension** is preserved by the network architecture itself; temporal compression requires the chunked inference loop with `feat_cache`
- **Reparameterization** splits `z_dim*2` channels into mu/log_var, samples `z = mu + std*eps`
- **Latent normalization** centers latents before DiT consumption: `z_norm = (z - mean) / std`

### Source References
| Symbol | Location |
|--------|----------|
| Encoder3d | `wan_video_vae.py`, line 517 |
| Encoder3d.forward | `wan_video_vae.py`, line 569 |
| VideoVAE_.forward (reparameterize) | `wan_video_vae.py`, line 978 |
| WanVideoVAE (normalization) | `wan_video_vae.py`, line 1058 |

## Exercises

### Exercise 1 -- Different dim_mult
Change `dim_mult` to `[1, 2, 4, 8]` (instead of `[1, 2, 4, 4]`). What would the channel count be at level 3? What happens to parameter count? Instantiate the encoder and verify.

### Exercise 2 -- More ResBlocks per level
The encoder uses `num_res_blocks=2` per level. If you increase to `num_res_blocks=4`, how does this affect the spatial output size? (Answer: it doesn't -- only Resample changes spatial dims. More ResBlocks increase depth and parameter count without changing the spatial downsampling structure.)

### Exercise 3 -- Production compression ratio
Calculate the theoretical compression ratio for a production encoder (`dim=96`, `z_dim=16`) processing a 480x832 video with 21 frames (with chunked temporal compression). Compare total input voxels to total latent voxels.
- Input: 3 channels * 21 frames * 480 * 832
- Latent: 16 channels * T_latent frames * 60 * 104 (spatial /8)
- T_latent = 1 + (21-1)//4 = 6